# GP09 — Spatial Join

Σύνδεσε σημεία υπηρεσιών με τη διοικητική περιοχή όπου βρίσκονται.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = Path('data/gp09')
df = pd.read_csv(root / 'services.csv')
services = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs='EPSG:4326')
districts = gpd.read_file(root / 'districts.geojson')


Τα σημεία είναι το left dataset και οι περιοχές το right dataset. Χρησιμοποίησε predicate `within`.

In [ ]:
joined = gpd.sjoin(services, districts, how='left', predicate='within')
joined.head()


In [ ]:
counts = joined.dropna(subset=['district_id']).groupby('district_name').size()
print(counts)


QA: σύγκρινε counts πριν/μετά, unmatched rows, CRS και geometry retention. Εξήγησε αν το αποτέλεσμα είναι one-to-one ή one-to-many.

In [ ]:
assert len(joined) >= len(services)
assert joined.geometry.geom_type.eq('Point').all()
assert joined.crs.to_epsg() == 4326
print('unmatched:', joined['district_id'].isna().sum())


In [ ]:
joined.to_file('gp09_services_by_district.gpkg', layer='services', driver='GPKG')
counts.to_csv('gp09_service_counts.csv')
print('export complete')
